In [2]:
%pip install torch torchvision scikit-learn pandas pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 34.8 MB/s  0:00:03 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 32.7 MB/s  0:00:005.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 31.8 MB/s  0:00:00 eta 0:00:010:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 39.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 16.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [scikit-learn] 13/14 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, shutil, random, hashlib, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score, accuracy_score
import pandas as pd

# ============ CONFIG (keep identical across ALL models) ============
SEED = 42
data_path = '/Users/sujita/Downloads/drive-download-20261005T003625Z-1-001'  # <-- update this
output_dir = "dataset_split"
manifest_path = "split_manifest.csv"   # reuse this file for MobileNetV2 + EfficientNet B0
train_ratio, val_ratio = 0.70, 0.15    # test = remaining 0.15
batch_size = 32
num_epochs = 10
VALID_EXT = ('.jpg', '.jpeg', '.png')  # HEIC dropped: PIL can't read it by default

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============ STEP 1: Clean + remove duplicates ============
def clean_hidden(path):
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.startswith('.'):
                os.remove(os.path.join(root, f))

def file_hash(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

clean_hidden(data_path)

seen = {}          # hash -> first path seen (across ALL classes)
class_files = {}   # class -> list of unique image paths
dupes_removed = 0

for cls in sorted(os.listdir(data_path)):
    cls_path = os.path.join(data_path, cls)
    if not os.path.isdir(cls_path):
        continue
    class_files[cls] = []
    for fname in sorted(os.listdir(cls_path)):
        if not fname.lower().endswith(VALID_EXT):
            continue
        fpath = os.path.join(cls_path, fname)
        h = file_hash(fpath)
        if h in seen:
            dupes_removed += 1      # skip duplicate (not deleted from your original folder)
            continue
        seen[h] = fpath
        class_files[cls].append(fpath)

for cls, files in class_files.items():
    if len(files) < 90:
        print(f"⚠️ {cls}: only {len(files)} unique images")
print(f"Duplicates removed: {dupes_removed}")
print(f"Classes: {len(class_files)} | Unique images: {sum(len(v) for v in class_files.values())}")

# ============ STEP 2: Train / Val / Test split (saved to manifest) ============
if os.path.exists(manifest_path):
    manifest = pd.read_csv(manifest_path)
    print(f"Loaded existing split from {manifest_path}")
else:
    rows = []
    rng = random.Random(SEED)
    for cls, files in class_files.items():
        files = files[:]
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        for i, f in enumerate(files):
            split = "train" if i < n_train else "val" if i < n_train + n_val else "test"
            rows.append({"path": f, "class": cls, "split": split})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(manifest_path, index=False)
    print(f"Created new split -> {manifest_path}")

# Rebuild folders from scratch so no stale files remain
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
for _, r in manifest.iterrows():
    dest = os.path.join(output_dir, r["split"], r["class"])
    os.makedirs(dest, exist_ok=True)
    shutil.copy(r["path"], os.path.join(dest, os.path.basename(r["path"])))

print(manifest["split"].value_counts().to_string())

# ============ STEP 3: Load data ============
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(f"{output_dir}/train", transform=transform)
val_dataset   = datasets.ImageFolder(f"{output_dir}/val",   transform=transform)
test_dataset  = datasets.ImageFolder(f"{output_dir}/test",  transform=transform)
class_names = train_dataset.classes
num_classes = len(class_names)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# ============ STEP 4: Build ResNet50 ============
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ============ STEP 5: Train (validation used ONLY for model selection) ============
def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total

history, best_val_acc, best_state = [], 0.0, None
for epoch in range(1, num_epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = copy.deepcopy(model.state_dict())
    history.append({"Epoch": epoch, "Train Loss": round(tr_loss, 4), "Train Acc": round(tr_acc, 4),
                    "Val Loss": round(va_loss, 4), "Val Acc": round(va_acc, 4)})
    print(f"Epoch {epoch}/{num_epochs} | Train {tr_loss:.4f}/{tr_acc:.4f} | Val {va_loss:.4f}/{va_acc:.4f}")

print(pd.DataFrame(history))
model.load_state_dict(best_state)   # best epoch chosen on validation only

# ============ STEP 6: Final evaluation on HELD-OUT TEST set (run once) ============
y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        y_true.extend(labels.numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())

test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average='weighted')
print(f"\nTEST Accuracy: {test_acc:.4f} | TEST Weighted F1: {test_f1:.4f}")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# ============ STEP 7: Speed + size ============
def sync():
    if device.type == "cuda": torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()

dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    for _ in range(10):          # warm-up
        model(dummy)
    sync()
    start = time.time()
    for _ in range(100):
        model(dummy)
    sync()
speed_ms = (time.time() - start) / 100 * 1000
print(f"Inference Speed: {speed_ms:.2f} ms")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
size_mb = (total_params * 4) / (1024 ** 2)
print(f"Total Params: {total_params:,} | Trainable: {trainable_params:,} | Size: {size_mb:.2f} MB")

torch.save(model.state_dict(), "resnet50_model.pth")

Duplicates removed: 28
Classes: 73 | Unique images: 7315
Created new split -> split_manifest.csv
split
train    5115
test     1112
val      1088
Running on: mps
Epoch 1/10 | Train 2.4206/0.6717 | Val 1.2366/0.8667
Epoch 2/10 | Train 0.7475/0.9322 | Val 0.6950/0.9090
Epoch 3/10 | Train 0.4028/0.9627 | Val 0.5059/0.9237
Epoch 4/10 | Train 0.2597/0.9791 | Val 0.4163/0.9347
Epoch 5/10 | Train 0.1798/0.9900 | Val 0.3686/0.9338
Epoch 6/10 | Train 0.1275/0.9949 | Val 0.3378/0.9329
Epoch 7/10 | Train 0.0960/0.9984 | Val 0.3125/0.9403
Epoch 8/10 | Train 0.0743/0.9984 | Val 0.2923/0.9449
Epoch 9/10 | Train 0.0608/0.9994 | Val 0.2844/0.9449
Epoch 10/10 | Train 0.0480/1.0000 | Val 0.2731/0.9458
   Epoch  Train Loss  Train Acc  Val Loss  Val Acc
0      1      2.4206     0.6717    1.2366   0.8667
1      2      0.7475     0.9322    0.6950   0.9090
2      3      0.4028     0.9627    0.5059   0.9237
3      4      0.2597     0.9791    0.4163   0.9347
4      5      0.1798     0.9900    0.3686   0.9338
5 